In [12]:
import pandas as pd

chexpert = pd.read_csv("mimic-cxr-2.0.0-chexpert-LABEL.csv")
metadata = pd.read_csv("mimic-cxr-2.0.0-metadata-IMAGE INFO.csv")
split    = pd.read_csv("mimic-cxr-2.0.0-split.csv")

print("chexpert:", chexpert.shape)
print("metadata:", metadata.shape)
print("split:   ", split.shape)
print("\nchexpert columns:", chexpert.columns.tolist())

chexpert: (227827, 16)
metadata: (377110, 12)
split:    (377110, 4)

chexpert columns: ['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices']


In [13]:
target_labels = ['Pneumonia', 'Pneumothorax', 'Lung Lesion', 'Lung Opacity']

chexpert_filtered = chexpert[['subject_id', 'study_id'] + target_labels].copy()

# 只排除含有 -1 的 row，NaN 保留（之後當 0 處理）
has_uncertain = (chexpert_filtered[target_labels] == -1).any(axis=1)
chexpert_filtered = chexpert_filtered[~has_uncertain].copy()

# NaN → 0
chexpert_filtered[target_labels] = chexpert_filtered[target_labels].fillna(0)

print(f"排除 -1 後剩下: {len(chexpert_filtered)} studies")
print("\n正樣本數量 per label:")
print((chexpert_filtered[target_labels] == 1).sum())

排除 -1 後剩下: 204349 studies

正樣本數量 per label:
Pneumonia       15766
Pneumothorax     9811
Lung Lesion      5394
Lung Opacity    38171
dtype: int64


In [14]:
has_positive = (chexpert_filtered[target_labels] == 1).any(axis=1)
all_negative = (chexpert_filtered[target_labels] == 0).all(axis=1)

selected = chexpert_filtered[has_positive | all_negative].copy()
print(f"Selected studies: {len(selected)}")

Selected studies: 204349


In [15]:
df = selected.merge(split[['dicom_id', 'study_id', 'subject_id', 'split']],
                    on=['study_id', 'subject_id'], how='inner')

df = df.merge(metadata[['dicom_id', 'ViewPosition']],
              on='dicom_id', how='inner')

print(f"After merge: {len(df)} images")
print(df['ViewPosition'].value_counts())

After merge: 339989 images
ViewPosition
AP                129930
PA                 88503
LATERAL            75134
LL                 32066
PA LLD                 4
RAO                    2
LAO                    2
AP AXIAL               2
AP RLD                 2
SWIMMERS               1
AP LLD                 1
XTABLE LATERAL         1
PA RLD                 1
LPO                    1
Name: count, dtype: int64


In [16]:
df_pa = df[df['ViewPosition'] == 'PA'].copy()

def build_path(row):
    p_prefix = f"p{str(row['subject_id'])[:2]}"
    return f"files/{p_prefix}/p{row['subject_id']}/s{row['study_id']}/{row['dicom_id']}.jpg"

df_pa['image_path'] = df_pa.apply(build_path, axis=1)

# 儲存完整資料表
df_pa.to_csv("selected_dataset.csv", index=False)

# 儲存下載清單
df_pa['image_path'].to_csv("images_to_download.txt", index=False, header=False)

print(f"PA images total: {len(df_pa)}")
print(f"\nSplit 分布:")
print(df_pa['split'].value_counts())
print(df_pa.head(3))

PA images total: 88503

Split 分布:
split
train       86931
test          876
validate      696
Name: count, dtype: int64
   subject_id  study_id  Pneumonia  Pneumothorax  Lung Lesion  Lung Opacity  \
0    10000032  50414267        0.0           0.0          0.0           0.0   
2    10000032  53189527        0.0           0.0          0.0           0.0   
8    10000898  50771383        0.0           0.0          0.0           0.0   

                                       dicom_id  split ViewPosition  \
0  02aa804e-bde0afdd-112c0b34-7bc16630-4e384014  train           PA   
2  2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab  train           PA   
8  2a280266-c8bae121-54d75383-cac046f4-ca37aa16  train           PA   

                                          image_path  
0  files/p10/p10000032/s50414267/02aa804e-bde0afd...  
2  files/p10/p10000032/s53189527/2a2277a9-b0ded15...  
8  files/p10/p10000898/s50771383/2a280266-c8bae12...  


In [17]:
print("各 label 正樣本數 (PA only):")
print((df_pa[target_labels] == 1).sum())
print(f"\n總共: {len(df_pa)} 張")
print(f"至少有一個正樣本: {(df_pa[target_labels] == 1).any(axis=1).sum()} 張")
print(f"全陰性: {(df_pa[target_labels] == 0).all(axis=1).sum()} 張")

各 label 正樣本數 (PA only):
Pneumonia       5534
Pneumothorax    2263
Lung Lesion     2696
Lung Opacity    9494
dtype: int64

總共: 88503 張
至少有一個正樣本: 15919 張
全陰性: 72584 張


In [18]:
positive_df = df_pa[(df_pa[target_labels] == 1).any(axis=1)]
negative_df = df_pa[(df_pa[target_labels] == 0).all(axis=1)]

# 負樣本保留正樣本的 2 倍
neg_sample = negative_df.sample(n=len(positive_df) * 2, random_state=42)

df_balanced = pd.concat([positive_df, neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced 後總張數: {len(df_balanced)}")
print(f"  正樣本: {len(positive_df)}")
print(f"  負樣本: {len(neg_sample)}")
print(f"\nSplit 分布:")
print(df_balanced['split'].value_counts())

# 儲存
df_balanced.to_csv("selected_dataset_balanced.csv", index=False)
df_balanced['image_path'].to_csv("images_to_download_balanced.txt", index=False, header=False)
print("\n✅ 已儲存 selected_dataset_balanced.csv")

Balanced 後總張數: 47757
  正樣本: 15919
  負樣本: 31838

Split 分布:
split
train       46868
test          533
validate      356
Name: count, dtype: int64

✅ 已儲存 selected_dataset_balanced.csv


In [22]:
# Cell 8：簡單抽 0.1%
df_small = df_balanced.sample(frac=0.1, random_state=42).reset_index(drop=True)

print(f"小樣本總張數: {len(df_small)}")
print(f"\n各 label 正樣本數:")
print((df_small[target_labels] == 1).sum())

# 儲存
df_small.to_csv("selected_dataset_small.csv", index=False)
df_small['image_path'].to_csv("images_to_download_small.txt", index=False, header=False)
print("\n✅ 已儲存 selected_dataset_small.csv")

小樣本總張數: 4776

各 label 正樣本數:
Pneumonia       530
Pneumothorax    211
Lung Lesion     284
Lung Opacity    945
dtype: int64

✅ 已儲存 selected_dataset_small.csv


In [ ]:
import requests, os
from tqdm import tqdm

# Login
session = requests.Session()
login_url = "https://physionet.org/login/"

r = session.get(login_url)
csrf = r.cookies.get("csrftoken")

login_data = {
    "username": "placeholder",
    "password": "placeholder",
    "csrfmiddlewaretoken": csrf
}

r = session.post(login_url, data=login_data, headers={"Referer": login_url})
print(f"Login status: {r.status_code}")

# Download
base_url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
save_dir = "images"
os.makedirs(save_dir, exist_ok=True)

with open("images_to_download_small.txt") as f:
    paths = [l.strip() for l in f.readlines() if l.strip()]

print(f"Starting download: {len(paths)} images...")
failed = []

for path in tqdm(paths):
    local_path = os.path.join(save_dir, path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if os.path.exists(local_path):
        continue
    r = session.get(base_url + path)
    if r.status_code == 200:
        with open(local_path, 'wb') as f:
            f.write(r.content)
    else:
        failed.append((path, r.status_code))

print(f"Done! Failed: {len(failed)} images")
if failed:
    print("First 5 failed:", failed[:5])

Login status: 200
Starting download: 48 images...


  2%|▏         | 1/48 [00:45<35:56, 45.89s/it]


KeyboardInterrupt: 